<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m5_01_classification_trees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M5.01 — Red or white?

In the lab we trained a decision tree classifier, read it as a list of
questions, and compared its border with that of a logistic regression.
In this workbook you do the same **on chemical analyses of wine**. You
will compute the impurity a tree tries to reduce, discover something
trees do not care about at all, and find the kind of border they are bad
at.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "better_one_question_tree": "70c154ecaf850415",
    "gini_half_half": "d2cbad71ff333de6",
    "gini_right_group": "d2cbad71ff333de6",
    "max_leaves_depth_2": "4b227777d4dd1fc6",
    "tree_border": "750e1381fee0e23a",
    "tree_needs_scaling": "9390298f3fb0c5b1"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and target

Each row is one Portuguese wine analysed in a laboratory: 11
measurements such as acidity, sugar, chlorides (salt), sulfur dioxide
(a preservative) and alcohol. `color` says whether the wine is red or
white, and `quality` is a mark from 0 to 10 given by tasters (we use it
in workbook M5.03).

Can a model tell **red from white** from the chemistry alone?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

wine = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/wine_quality.csv"
)
target = wine["color"]
print(wine["color"].value_counts())
wine.head()

To be able to draw what the models do, we start with two measurements:
`total_sulfur_dioxide` and `chlorides`.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

two_features = ["total_sulfur_dioxide", "chlorides"]
data = wine[two_features]
data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state=0, stratify=target
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

## 2. A tree is a list of questions

A decision tree asks yes/no questions about **one feature at a time**
("is `total_sulfur_dioxide` below 67.5?"). `max_depth` is the number of
questions it may ask in a row.

✏️ A tree with `max_depth=2` asks two questions in a row. Into how many
groups (leaves) can it split the wines **at most**?

In [ ]:
max_leaves_depth_2 = ____
check("max_leaves_depth_2", max_leaves_depth_2, hint="Each question splits a group in two.")

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(data_train, target_train)
fig, ax = plt.subplots(figsize=(11, 4.5))
_ = plot_tree(tree, feature_names=two_features, class_names=list(tree.classes_), filled=True, impurity=True, ax=ax, fontsize=9)

Each box is a group of wines. `value` gives the number of red and white
wines in it, and `gini` measures how **mixed** the group is.

The **Gini impurity** of a group is `1 − (share of red)² − (share of
white)²`. A group with only one colour has an impurity of 0. At each
step, the tree chooses the question that makes the two new groups as
pure as possible.

✏️ A group holds as many red wines as white wines. What is its Gini
impurity?

In [ ]:
gini_half_half = ____
check("gini_half_half", gini_half_half, hint="Both shares are 0.5. Apply the formula.")

With two classes, 0.5 is the worst possible value: the group tells us
nothing.

## 3. The border of a tree and the border of a line

The two pictures show, for every combination of the two measurements,
what each model predicts (red or blue background) and the test wines.

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

colors = target_test.map({"red": 1, "white": 0})
models = {
    "decision tree (depth 2)": tree,
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression()).fit(data_train, target_train),
}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (name, model) in zip(axes, models.items()):
    DecisionBoundaryDisplay.from_estimator(
        model, data_train, response_method="predict", alpha=0.4, cmap="coolwarm", ax=ax
    )
    ax.scatter(data_test.iloc[:, 0], data_test.iloc[:, 1], c=colors, cmap="coolwarm", s=6, edgecolor="black", linewidth=0.2)
    ax.set(title=f"{name}: test accuracy {model.score(data_test, target_test):.3f}", ylim=(0, 0.3), xlim=(0, 300))

✏️ The tree asks about one feature at a time. What can its border be
made of? Write `"horizontal and vertical segments"` or `"segments in any
direction"`.

In [ ]:
tree_border = ____
check("tree_border", tree_border)

## 4. ➕ Going further: something trees do not care about

In module 1 and in workbook M2.01, scaling changed everything for
nearest neighbours. `total_sulfur_dioxide` goes up to 440 and
`chlorides` stays below 1: very different scales.

🔮 **Predict first.** We train the same tree on **all 11 measurements**,
once as they are and once after a `StandardScaler`. For how many of the
test wines will the two trees give a **different** prediction? **"none"**,
**"a few percent"** or **"many"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["none", "a few percent", "many"])

In [ ]:
all_features = [name for name in wine.columns if name not in ("color", "quality")]
full_train, full_test, target_train, target_test = train_test_split(
    wine[all_features], target, random_state=0, stratify=target
)
plain_tree = DecisionTreeClassifier(random_state=0).fit(full_train, target_train)
scaled_tree = make_pipeline(StandardScaler(), DecisionTreeClassifier(random_state=0)).fit(full_train, target_train)
log_tree = DecisionTreeClassifier(random_state=0).fit(np.log1p(full_train), target_train)

n_different_scaled = (plain_tree.predict(full_test) != scaled_tree.predict(full_test)).sum()
n_different_log = (plain_tree.predict(full_test) != log_tree.predict(np.log1p(full_test))).sum()
print(f"test wines predicted differently after scaling:     {n_different_scaled} of {len(full_test)}")
print(f"test wines predicted differently after a logarithm: {n_different_log} of {len(full_test)}")

✏️ Does a decision tree need its features to be scaled? Write `"yes"` or
`"no"`.

In [ ]:
tree_needs_scaling = ____
check("tree_needs_scaling", tree_needs_scaling)

A question such as "is the value below 67.5?" only depends on the
**order** of the wines along a feature. Scaling, or even taking a
logarithm, moves the values and keeps their order: the tree asks the
same questions with other thresholds and sorts the wines into the same
groups. For trees, the scaler of the previous modules is unnecessary.

## 5. ➕ Going further: the border trees are bad at

Look at the two pictures of section 3 again. The red and white wines
are separated by a border that **leans**: the logistic regression draws
it with one line. The tree has to approximate a leaning border with a
staircase, and each step costs a question.

We compare trees that may ask a single question (`max_depth=1`):

- on the two original features;
- on **one new feature** that combines them. Both measurements are put
  on a comparable scale (logarithm, then standard scaling), and the new
  feature is their difference: "much sulfur dioxide **and** little
  salt".

🔮 **Predict first.** A single question on a single feature. Can the
tree with the combined feature beat the tree that has **both** original
features to choose from? `"yes"` or `"no"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["yes", "no"])

In [ ]:
standardised = pd.DataFrame(
    StandardScaler().fit_transform(np.log(data)), columns=two_features, index=data.index
)
combined = (standardised["total_sulfur_dioxide"] - standardised["chlorides"]).to_frame("sulfur_minus_salt")

one_question = DecisionTreeClassifier(max_depth=1, random_state=0)
accuracy = {
    "one question, two original features": cross_val_score(one_question, data, target, cv=cv).mean(),
    "one question, the combined feature": cross_val_score(one_question, combined, target, cv=cv).mean(),
    "logistic regression, two original features": cross_val_score(
        make_pipeline(StandardScaler(), LogisticRegression()), data, target, cv=cv
    ).mean(),
}
pd.Series(accuracy, name="accuracy").round(3).to_frame()

✏️ Which one-question tree is better? Write `"original features"` or
`"combined feature"`.

In [ ]:
better_one_question_tree = ____
check("better_one_question_tree", better_one_question_tree)

With the right combination, **one** question does as well as the
depth-2 tree of section 2. The information was in the two features all
along, but a tree cannot ask "is this one large **compared with** that
one?". This is the mirror image of workbook M4.02: there, a linear model
needed us to build interactions that a tree found by itself. Here, the
tree needs us to build a combination that a linear model handles
naturally.

🧠 **Think.** Which kind of model, tree or linear, would you try first
on a problem where the classes are separated by "feature A is larger
than feature B"? And where they are separated by "feature A is between
10 and 20"?

## 6. ➕ Going further: a split that makes things worse?

The tree always chooses the question that reduces impurity **on
average** over the two new groups. Here is the tree of section 2 with
one more level of questions. For each question we print the impurity of
the group before, and of the two groups after.

In [ ]:
deeper = DecisionTreeClassifier(max_depth=3, random_state=0).fit(data, target).tree_
rows = []
for node in range(deeper.node_count):
    left, right = deeper.children_left[node], deeper.children_right[node]
    if left != -1:
        rows.append(
            {
                "wines before": deeper.n_node_samples[node],
                "gini before": deeper.impurity[node],
                "wines left": deeper.n_node_samples[left],
                "gini left": deeper.impurity[left],
                "wines right": deeper.n_node_samples[right],
                "gini right": deeper.impurity[right],
            }
        )
splits = pd.DataFrame(rows)
splits.round(3)

In several rows, one of the two new groups is **more mixed** than the
group it came from.

✏️ A group contains 4 white wines and 1 red (gini 0.32). A question
sends 3 white wines to the left, and 1 white with the 1 red to the
right. What is the impurity of the **right** group?

In [ ]:
gini_right_group = ____
check("gini_right_group", gini_right_group, hint="The right group holds one wine of each colour.")

The right group (0.5) is worse than the parent (0.32), and the split is
still a good one: the left group is perfectly pure, and it is larger.
What the tree minimises is the impurity of the children **weighted by
their size**: here (3 × 0 + 2 × 0.5) / 5 = 0.2, less than 0.32.

## 7. ➕ Going further: missing measurements

Laboratories lose samples. Suppose that 10% of the `chlorides` values
are missing. Most scikit-learn models refuse data with missing values
and need an imputer first.

🔮 **Predict first.** Will a `DecisionTreeClassifier` accept the data
with missing values, without any imputer? **"it trains"** or **"it
raises an error"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["it trains", "it raises an error"])

In [ ]:
with_gaps = wine[all_features].copy()
missing = np.random.default_rng(0).random(len(with_gaps)) < 0.10
with_gaps.loc[missing, "chlorides"] = np.nan
print(f"{missing.sum()} wines without a chlorides value")

for name, model in [("decision tree", DecisionTreeClassifier(random_state=0)), ("logistic regression", LogisticRegression(max_iter=5000))]:
    try:
        scores = cross_val_score(model, with_gaps, target, cv=cv, error_score="raise")
        print(f"{name}: trains, accuracy {scores.mean():.3f}")
    except ValueError as error:
        print(f"{name}: error — {str(error).splitlines()[0]}")

Recent versions of scikit-learn let a tree handle missing values by
itself: at each question, the wines whose value is missing are sent to
the side where they help most. No invented value is needed.

🧠 **Think.** An imputer would replace the missing chlorides by, say,
the mean. What does the model then believe about those wines? What can
the tree do that the imputer cannot?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - how to read the picture of the tree: questions, `value`, `gini`;
   - why the tree gives the same predictions with and without scaling;
   - the leaning border: why one question on a combined feature beats
     two questions on the original features;
   - how a split can produce a group more mixed than its parent and
     still be a good split.